In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
import string
import numpy as np
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

train_df = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

In [3]:
print("--- Question 1: Frequency Distribution ---")
answer_counts = train_df['answer'].value_counts()
most_freq = answer_counts.max()
least_freq = answer_counts.min()
print(f"Counts:\n{answer_counts}")
print(f"Sum of most and least frequent: {most_freq + least_freq}")

--- Question 1: Frequency Distribution ---
Counts:
answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
Sum of most and least frequent: 814


In [4]:
print("\n--- Question 2: Vocabulary Size of Cleaned Prompts ---")
def clean_and_split(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return text.split()

all_words = []
for prompt in train_df['prompt']:
    all_words.extend(clean_and_split(prompt))
    
unique_words = set(all_words)
print(f"Total unique words in prompt column: {len(unique_words)}")


--- Question 2: Vocabulary Size of Cleaned Prompts ---
Total unique words in prompt column: 859


In [5]:
print("\n--- Question 3: Stop Words Filtering on Row ID 1 ---")
row_1_words = clean_and_split(train_df.loc[train_df['id'] == 1, 'prompt'].values[0])
filtered_words = [word for word in row_1_words if word not in ENGLISH_STOP_WORDS]
print(f"Words left after filtering Row ID 1: {len(filtered_words)}")


--- Question 3: Stop Words Filtering on Row ID 1 ---
Words left after filtering Row ID 1: 13


In [6]:
print("\n--- Question 4: TF-IDF Total Feature Columns ---")
all_text = []
for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    all_text.extend(train_df[col].dropna().astype(str).tolist())

vectorizer = TfidfVectorizer(stop_words='english')
vectorizer.fit(all_text)
print(f"Total TF-IDF feature columns (vocabulary size): {len(vectorizer.vocabulary_)}")


--- Question 4: TF-IDF Total Feature Columns ---
Total TF-IDF feature columns (vocabulary size): 2762


In [7]:
print("\n--- Question 5: Cosine Similarity for Row ID 1 ---")
row_1 = train_df[train_df['id'] == 1].iloc[0]
prompt_text = str(row_1['prompt'])
option_a_text = str(row_1['A'])

tfidf_matrix = vectorizer.transform([prompt_text, option_a_text])
sim_score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
print(f"Cosine similarity between prompt and Option A: {sim_score:.4f}")


--- Question 5: Cosine Similarity for Row ID 1 ---
Cosine similarity between prompt and Option A: 0.2328


In [8]:
print("\n--- Question 6: Baseline TF-IDF Accuracy ---")
correct_matches = 0
for _, row in train_df.iterrows():
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    
    scores = {}
    for label in ['A', 'B', 'C', 'D', 'E']:
        opt_vec = vectorizer.transform([str(row[label])])
        scores[label] = cosine_similarity(prompt_vec, opt_vec)[0][0]
        
    best_guess = max(scores, key=scores.get)
    if best_guess == row['answer']:
        correct_matches += 1

accuracy = (correct_matches / len(train_df)) * 100
print(f"Highest similarity matches correct answer: {accuracy:.2f}%")


--- Question 6: Baseline TF-IDF Accuracy ---
Highest similarity matches correct answer: 13.70%


In [9]:
print("\n--- Question 7 & 8: Understanding MAP@3 ---")
def apk(actual, predicted_list):
    if actual in predicted_list[:3]:
        index = predicted_list.index(actual)
        return 1.0 / (index + 1)
    return 0.0

print(f"MAP@3 for actual 'C' and prediction 'C A B': {apk('C', ['C', 'A', 'B'])}")
print(f"MAP@3 for actual 'B' and prediction 'D B E': {apk('B', ['D', 'B', 'E'])}")


--- Question 7 & 8: Understanding MAP@3 ---
MAP@3 for actual 'C' and prediction 'C A B': 1.0
MAP@3 for actual 'B' and prediction 'D B E': 0.5


In [10]:
print("\n--- Question 9: Majority Class Baseline MAP@3 ---")
top_3_labels = answer_counts.head(3).index.tolist()
print(f"Top 3 most frequent labels: {top_3_labels}")

majority_scores = [apk(row['answer'], top_3_labels) for _, row in train_df.iterrows()]
print(f"Majority Class Baseline MAP@3: {np.mean(majority_scores):.4f}")


--- Question 9: Majority Class Baseline MAP@3 ---
Top 3 most frequent labels: ['B', 'C', 'A']
Majority Class Baseline MAP@3: 0.4213


In [11]:
print("\n--- Question 10: TF-IDF Pipeline MAP@3 ---")
tfidf_map_scores = []
for _, row in train_df.iterrows():
    prompt_vec = vectorizer.transform([str(row['prompt'])])
    
    scores = {}
    for label in ['A', 'B', 'C', 'D', 'E']:
        opt_vec = vectorizer.transform([str(row[label])])
        scores[label] = cosine_similarity(prompt_vec, opt_vec)[0][0]
        
    top_3_preds = sorted(scores, key=scores.get, reverse=True)[:3]
    tfidf_map_scores.append(apk(row['answer'], top_3_preds))

print(f"TF-IDF Pipeline average MAP@3: {np.mean(tfidf_map_scores):.4f}")


--- Question 10: TF-IDF Pipeline MAP@3 ---
TF-IDF Pipeline average MAP@3: 0.3119
